In [39]:
# the code will try to read the tensorboard logs and extract results
import os
import seaborn as sns
import matplotlib.pyplot as plt
import torch

In [ ]:
import os
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

log_dir = "runs/"
rows = []

for root, _, files in os.walk(log_dir):
    for f in files:
        if "tfevents" in f:
            event_path = os.path.join(root, f)
            acc = EventAccumulator(event_path)
            acc.Reload()

            run_name = os.path.relpath(root, log_dir)
            scalar_tags = acc.Tags().get("scalars", [])

            for tag in scalar_tags:
                for e in acc.Scalars(tag):
                    rows.append({
                        "run": run_name,
                        "tag": tag,
                        "step": e.step,
                        "value": e.value,
                        "wall_time": e.wall_time,
                        "event_file": event_path,
                    })

df = pd.DataFrame(rows)
# df.to_excel("tensorboard_scalars.xlsx", index=False)
print("Saved to tensorboard_scalars.xlsx")

# Save plots from tensorboard

In [ ]:
os.makedirs("plots_by_tag", exist_ok=True)

for uniq_tag in df.tag.unique():
    tag_df = df[df.tag == uniq_tag]
    
    # plot a single plot for each unique tag
    plt.figure(figsize=(10, 6))   
    for experiment in tag_df.run.unique():
        exp_df = tag_df[tag_df.run == experiment]
        total_epochs = len(exp_df)
        
        dataset = experiment.split("_")[2]
        dataset = "MSD" if "task" in dataset.lower() else dataset.upper()
        organ = experiment.split("_")[-2]
        model = experiment.split("_")[-1]
        
        # check if model checkpoint file exists
        model_checkpoint_name = "_".join(experiment.split("_")[:2])
        model_checkpoint_path = os.path.join('model_checkpoints', model_checkpoint_name + ".pth")
        

        if not os.path.exists(model_checkpoint_path):
            print(f"{model_checkpoint_path=}")

        plt.plot(exp_df.step, exp_df.value, label=f"{dataset} {organ.capitalize()} {model.capitalize()}")

    plt.title(f"{uniq_tag.split('/')[-1]} {uniq_tag.split('/')[-2]}")
    plt.xlabel("Step / Epoch")
    plt.ylabel("Value")
    plt.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(f"plots_by_tag/{uniq_tag.split('/')[-1].lower()}_{uniq_tag.split('/')[-2].lower()}.png", dpi=300, bbox_inches='tight')

# Save excel tables 

In [ ]:
os.makedirs("tables_by_tag", exist_ok=True)

for uniq_tag in df.tag.unique():
    tag_df = df[df.tag == uniq_tag]
    
    dict_list = []
    for experiment in tag_df.run.unique():
        exp_df = tag_df[tag_df.run == experiment]
        exp_df.sort_values("step", inplace=True)
        total_epochs = len(exp_df)
        
        dataset = experiment.split("_")[2]
        dataset = "MSD" if "task" in dataset.lower() else dataset.upper()
        organ = experiment.split("_")[-2]
        organ = 'Hepatic vessel' if 'hepaticvessel' in organ.lower() else organ.capitalize()
        model = experiment.split("_")[-1]
        model = 'Davood' if 'davood' in model.lower() else '3D U-Net' if '3dunet' in model.lower() else 'SegResNet' if 'segresnet' in model.lower() else model
        
        # check if model checkpoint file exists
        model_checkpoint_name = "_".join(experiment.split("_")[:2])
        model_checkpoint_path = os.path.join('model_checkpoints', model_checkpoint_name + ".pth")

        stage = uniq_tag.split("/")[-1]
        parameter = uniq_tag.split("/")[-2]

        dict_list.append({
            "Dataset": dataset, 
            "Organ": organ, 
            "Model": model, 
            "Checkpoint path": model_checkpoint_path,
            "Stage": stage,
            "Epochs": total_epochs,
            f"{parameter}": round(exp_df.value.iloc[-1], 2) if len(exp_df) > 0 else None
            })

        if not os.path.exists(model_checkpoint_path):
            print(f"{model_checkpoint_path=}")

    pd.DataFrame(dict_list).to_excel(f"tables_by_tag/{uniq_tag.split('/')[-1].lower()}_{uniq_tag.split('/')[-2].lower()}.xlsx", index=False)


# PTH model checkpoints

In [79]:
all_runs = df.run.unique()
runs_all_datasets = [i for i in all_runs if 'all' in i]
runs_all_datasets

dice_test_name = "dice_list_test"
dice_val_name = "dice_list_val"
batch_test_name = "info_batch_test"
batch_val_name = "info_batch_val"

all_datasets_model_split_by_organ = []
for all_dataset in runs_all_datasets:
    model_checkpoint_name = "_".join(all_dataset.split("_")[:2])
    model = all_dataset.split("_")[-1]
    checkpoint_path = os.path.join("model_checkpoints", model_checkpoint_name + ".pth")

    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    dict_list = ckpt['model_config']
    

    for iou_val, batch_dict in zip(dict_list[dice_test_name], dict_list[batch_test_name]):
        dataset = batch_dict['dataset'][0]
        organ = batch_dict['object'][0]
        case_number = int(batch_dict['case_number'][0])
        all_datasets_model_split_by_organ.append({'model': model, 'stage': 'test', 'dataset': dataset, 'organ': organ, 'case_number': case_number, 'iou': iou_val})

    for iou_val, batch_dict in zip(dict_list[dice_val_name], dict_list[batch_val_name]):
        dataset = batch_dict['dataset'][0]
        organ = batch_dict['object'][0]
        case_number = int(batch_dict['case_number'][0])
        all_datasets_model_split_by_organ.append({'model': model, 'stage': 'val', 'dataset': dataset, 'organ': organ, 'case_number': case_number, 'iou': iou_val})

# Use the models trained on all the organs and calculate the average dice value per organ

In [ ]:
all_datasets_model_split_by_organ_df = pd.DataFrame(all_datasets_model_split_by_organ)
uniq_models = all_datasets_model_split_by_organ_df['model'].unique()
uniq_stages = all_datasets_model_split_by_organ_df['stage'].unique()
uniq_organs = all_datasets_model_split_by_organ_df['organ'].unique()


mean_dice_all_models_split_by_organ_list = []
for uniq_model in uniq_models:
    model_df = all_datasets_model_split_by_organ_df[all_datasets_model_split_by_organ_df['model'] == uniq_model]

    for uniq_stage in uniq_stages:
        model_stage_df = model_df[model_df['stage'] == uniq_stage]

        for uniq_organ in uniq_organs:
            model_stage_organ_df = model_stage_df[model_stage_df['organ'] == uniq_organ]
            mean_dice = model_stage_organ_df['iou'].mean()
            mean_dice_all_models_split_by_organ_list.append({'model': uniq_model, 'stage': uniq_stage, 'organ': uniq_organ, 'mean_dice': mean_dice})

df_final = pd.DataFrame(mean_dice_all_models_split_by_organ_list)

df_final.to_excel("models_trained_on_all_datasets_averaged_by_organ.xlsx", index=False)
